**Normalization (Standardization)**

Normalization, also known as standardization, ensures that the data follows a standard distribution with a mean (μ) of 0 and a standard deviation (σ) of 1.

The formula used for Z-score normalization is:

𝑍=(𝑋−𝜇)/𝜎

Where:

X is the original value,

μ is the mean of the values,

σ (sigma) is the standard deviation

**Applying Normalization in Batch Normalization :**

Instead of applying normalization to the input data, Batch Normalization applies this transformation to the outputs of neurons (perceptrons) inside the layers of the neural network.

It calculates the mean (μ) and standard deviation (σ) from the activations of a layer.

Then, it normalizes these activations using the Mean / Standard Deviation formula.

Why Use Batch Normalization?

Batch Normalization helps improve:

✅ Training stability by reducing internal covariate shift

✅ Faster convergence by allowing higher learning rates

✅ Better generalization by reducing dependence on initialization

**Gamma (γ) and Beta (β) Parameters**
After normalization, learnable parameters γ (gamma) and β (beta) allow the network to adjust the final output.

γ (gamma): Controls the scaling of activations.

β (beta): Controls the shifting of activations.

These parameters help the network retain flexibility and avoid being overly constrained by strict normalization

In [10]:
import tensorflow as tf
from tensorflow import keras

In [11]:
# Load Fashion MNIST data and split it into training and test data
(X_train_full, y_train_full), (X_test, y_test) = keras.datasets.fashion_mnist.load_data()

# Normalize data : to make values ​​between 0 and 1 to improve model performance
X_train_full = X_train_full / 255.0
X_test = X_test / 255.0

# Split the training data into a training set (55,000) and a validation set (5,000)
# X_train_full contains 60,000 images for training | X_test contains 10,000 images for testing
# The first 5000 samples of X_train_full are for the validation set, and the rest are for the training set:
# X_valid contains 5,000 samples (from 0 to 4,999)
# X_train contains 60,000 - 5,000 = 55,000 samples (from 5,000 to 59,999)
X_valid, X_train = X_train_full[:5000], X_train_full[5000:]
y_valid, y_train = y_train_full[:5000], y_train_full[5000:]

29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [12]:
# Building the neural model
model = keras.models.Sequential([
    keras.layers.Flatten(input_shape=[28,28]), # Flatten the 28x28 image into a 1D vector
    keras.layers.BatchNormalization(),  # Apply Batch Normalization to stabilize inputs
    keras.layers.Dense(300,activation='relu'), # First hidden layer with 300 neurons and ReLU activation
    keras.layers.BatchNormalization(), # Apply Batch Normalization to the hidden layer
    keras.layers.Dense(100,activation='relu'), # Second hidden layer with 100 neurons and ReLU activation
    keras.layers.BatchNormalization(),  # Apply Batch Normalization to improve learning stability
    keras.layers.Dense(10,activation='softmax') # Output layer with 10 neurons for classification
    ])

In [13]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ flatten_1 (Flatten)                  │ (None, 784)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_3                │ (None, 784)                 │           3,136 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_3 (Dense)                      │ (None, 300)                 │         235,500 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_4                │ (None, 300)                 │           1,200 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_4 (Dense)                      │ (None, 100)                 │          30,100 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_5                │ (None, 100)                 │             400 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_5 (Dense)                      │ (None, 10)                  │           1,010 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 271,346 (1.04 MB)

 Trainable params: 268,978 (1.03 MB)

 Non-trainable params: 2,368 (9.25 KB)

 Trainable params: 268,978 (1.03 MB) => gamma and beta : Used to make the neural network have the ability to control the layers

 Non-trainable params: 2,368 (9.25 KB) => Mean and Standard division

In [17]:
bn1 = model.layers[1]  # Get the first Batch Normalization layer
# Check trainable variables of Batch Normalization layer
[(var.name,var.trainable) for var in bn1.variables]
# Lists the variables in the first Batch Normalization layer and whether they are trainable

[('gamma', True),
 ('beta', True),
 ('moving_mean', False),
 ('moving_variance', False)]

In [15]:
model.compile(loss='sparse_categorical_crossentropy',
              optimizer=keras.optimizers.SGD(learning_rate=1e-3),
              metrics=['accuracy'])

In [16]:
history = model.fit(X_train, y_train, epochs=10,
                    validation_data=(X_valid, y_valid))

Epoch 1/10
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 15s 8ms/step - accuracy: 0.6082 - loss: 1.2016 - val_accuracy: 0.8148 - val_loss: 0.5549
Epoch 2/10
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 13s 7ms/step - accuracy: 0.7959 - loss: 0.5915 - val_accuracy: 0.8408 - val_loss: 0.4791
Epoch 3/10
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 13s 7ms/step - accuracy: 0.8162 - loss: 0.5229 - val_accuracy: 0.8514 - val_loss: 0.4414
Epoch 4/10
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 22s 8ms/step - accuracy: 0.8306 - loss: 0.4840 - val_accuracy: 0.8632 - val_loss: 0.4185
Epoch 5/10
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - accuracy: 0.8393 - loss: 0.4603 - val_accuracy: 0.8650 - val_loss: 0.4042
Epoch 6/10
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 21s 7ms/step - accuracy: 0.8460 - loss: 0.4420 - val_accuracy: 0.8692 - val_loss: 0.3944
Epoch 7/10
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 21s 8ms/step - accuracy: 0.8502 - loss: 0.4256 - val_accuracy: 0.8712 - val_loss: 0.3846
Epoch 8/10
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 20s 8ms/step - accuracy: 0.8556 - loss: 0

The model is built using Sequential, including Batch Normalization to stabilize training

It is compiled with the SGD optimizer and cross-entropy loss for classification.

The model is trained for 10 epochs, with validation to monitor its performance.


---
⚖️ **Comparison Between Batch Normalization and SELU:**

We observe that Batch Normalization (BN) outperforms SELU in terms of accuracy.

In the SELU model, we trained 100 layers for 5 epochs, and the final accuracy was 0.7932.

In the Batch Normalization model, we used only 2 hidden layers, trained for 10 epochs, and achieved a higher accuracy of 0.8558.

Why is Batch Normalization More Effective?

SELU helps reduce the vanishing and exploding gradient problems, but it only minimizes them within a certain range—it does not eliminate them completely.

Batch Normalization, on the other hand, is more efficient in addressing these issues.

Evidence from Training Results
After 5 epochs in the Batch Normalization model, the accuracy reached 0.8393.

Comparing this to the 5th epoch of SELU, which had an accuracy of 0.7932, we see that Batch Normalization performs better even with fewer layers.